# PathSnap - Exploratory Data Analysis (EDA)

**Hackathon:** PathSnap GPS trajectory map-matching (UE24CS352A), 18 Nov 2026
**Member 1 deliverable:** dataset exploration and understanding

Each trip folder contains four files:
- `<id>.track` - noisy GPS trace, one `longitude latitude timestamp` row per sample
- `<id>.nodes` - road-network nodes, one `longitude latitude` row per node (node id = line index)
- `<id>.arcs` - directed road arcs as `from_node to_node` (indices into the nodes file)
- `<id>.route` - ground-truth route: ordered list of arc ids the vehicle actually travelled

Splits: `train/` (80 trips) and `student_test/` (10 trips).

In [ ]:
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# Point this at the dataset root (the folder containing train/ and student_test/)
DATA = "data"   # e.g. "Student folder" if the notebook sits next to it

def trip_dirs(split):
    base = os.path.join(DATA, split)
    return sorted(os.path.join(base, d) for d in os.listdir(base))

def read_track(path):
    pts, malformed, invalid = [], 0, 0
    with open(path) as f:
        for line in f:
            p = line.strip().split("\t")
            if len(p) != 3: p = line.split()
            if len(p) != 3: malformed += 1; continue
            try: lon, lat, ts = float(p[0]), float(p[1]), float(p[2])
            except ValueError: malformed += 1; continue
            if not (-180 <= lon <= 180 and -90 <= lat <= 90): invalid += 1; continue
            pts.append((lon, lat, ts))
    return (np.array(pts) if pts else np.zeros((0, 3))), malformed, invalid

def read_nodes(path):
    return np.array([tuple(map(float, (l.split("\t") if "\t" in l else l.split()))) for l in open(path) if l.strip()])

def read_arcs(path):
    out = []
    for l in open(path):
        if l.strip():
            p = l.split("\t") if "\t" in l else l.split()
            out.append((int(p[0]), int(p[1])))
    return np.array(out, dtype=np.int64)

def read_route(path):
    return [int(x) for x in open(path).read().split()] if os.path.exists(path) else None

## 1. Dataset overview

**Computed on all 90 trips (80 train + 10 test):**

| Quantity | min | median | mean | max |
|---|---|---|---|---|
| GPS points per trip | 262 | 1,496 | 2,529 | 19,313 |
| Road-network nodes per trip | 299 | 9,305 | 22,491 | 155,003 |
| Directed arcs per trip | 649 | 18,762 | 45,133 | 307,392 |
| Ground-truth route length (arcs, train) | 59 | 280 | 392 | 2,266 |
| Track duration (s) | 265 | 1,710 | 5,274 | 154,030 |

- Total GPS observations: **227,577**
- Every trip ships all four files; **all 90 trips include a non-empty ground-truth `.route`** (including the 10 `student_test` trips in this copy - worth confirming with the organizers whether that is intentional).
- Trips come from **many different cities worldwide** (mostly Europe, some North America and Asia); each trip has its own road network covering the region around the trace.

In [ ]:
rows = []
for split in ["train", "student_test"]:
    for d in trip_dirs(split):
        tid = os.path.basename(d)
        track, mal, inv = read_track(os.path.join(d, tid + ".track"))
        nodes = read_nodes(os.path.join(d, tid + ".nodes"))
        arcs  = read_arcs(os.path.join(d, tid + ".arcs"))
        route = read_route(os.path.join(d, tid + ".route"))
        ts = track[:, 2]
        rows.append(dict(split=split, trip=tid, gps_points=len(track), nodes=len(nodes),
                         arcs=len(arcs), route_len=len(route) if route else None,
                         malformed=mal, invalid=inv, duration_s=ts[-1]-ts[0]))
df = pd.DataFrame(rows)
df.describe().round(1)

## 2. Missing / invalid value analysis

**Result: the dataset is very clean.**
- Malformed lines: **0** across all 90 trips
- Coordinates outside valid lon/lat range: **0**
- Non-monotonic timestamps: **0**
- Consecutive duplicate GPS fixes: **6** in total (negligible)

So no imputation or coordinate repair is needed; preprocessing effort goes into map-matching features, not cleaning.

In [ ]:
print("malformed lines total:", df.malformed.sum())
print("invalid coords total:", df.invalid.sum())

## 3. Sampling rate / timestamp gaps

The documentation claims ~97% of samples arrive at 1 Hz. **Verified: 96.68% of the 227,487 consecutive-point intervals are exactly 1 second** (median gap = 1.0 s, mean = 2.09 s).
- 98.65% of intervals are <= 2 s
- Only 0.55% of intervals exceed 5 s - but a few trips contain very long gaps (max ~153,064 s, i.e. a multi-hour pause inside one trace)

Implication: distance/speed features computed between consecutive points are reliable most of the time, but the pipeline must handle occasional large gaps (e.g. reset the HMM or down-weight transition evidence across a big gap).

In [ ]:
gaps = []
for split in ["train", "student_test"]:
    for d in trip_dirs(split):
        tid = os.path.basename(d)
        t, _, _ = read_track(os.path.join(d, tid + ".track"))
        gaps.append(np.diff(t[:, 2]))
gaps = np.concatenate(gaps)
print(f"intervals={len(gaps):,}  pct==1s: {100*np.mean(gaps==1):.2f}%  pct<=2s: {100*np.mean(gaps<=2):.2f}%  max: {gaps.max():.0f}s")

plt.figure(figsize=(8,4.5))
plt.hist(gaps[gaps<=10], bins=np.arange(0,10.5,0.5), color="steelblue")
plt.axvline(1.0, color="crimson", ls="--", label="1 s (1 Hz)")
plt.title(f"Sampling interval distribution ({len(gaps):,} intervals)")
plt.xlabel("gap (seconds)"); plt.ylabel("count"); plt.legend(); plt.show()

## 4. Road-network connectivity (directed graph)

Analysed with union-find over each trip's directed arc list:
- Weakly connected components per trip: median **19** (min 1, max 421)
- The largest component covers a median **98.0%** of nodes (min 69.3%) - the network is essentially one connected region plus small dangling fragments
- Dead ends (out-degree 0): median **5.5** nodes/trip (max 81)
- Sources (in-degree 0): median **5** nodes/trip
- All ground-truth `.route` arc ids are valid indices into their trip's `.arcs` file (checked on all 90 trips)

Implication for the HMM: transition probabilities must respect arc direction and connectivity, and candidate roads should be looked up inside the main component; tiny disconnected fragments can be ignored.

In [ ]:
class UF:
    def __init__(self, n): self.p=list(range(n)); self.sz=[1]*n
    def find(self,x):
        while self.p[x]!=x: self.p[x]=self.p[self.p[x]]; x=self.p[x]
        return x
    def union(self,a,b):
        ra,rb=self.find(a),self.find(b)
        if ra==rb: return
        if self.sz[ra]<self.sz[rb]: ra,rb=rb,ra
        self.p[rb]=ra; self.sz[ra]+=self.sz[rb]

stats=[]
for split in ["train","student_test"]:
    for d in trip_dirs(split):
        tid=os.path.basename(d)
        nodes=read_nodes(os.path.join(d,tid+".nodes")); arcs=read_arcs(os.path.join(d,tid+".arcs"))
        uf=UF(len(nodes))
        for a,b in arcs: uf.union(int(a),int(b))
        comps=Counter(uf.find(i) for i in range(len(nodes)))
        out=np.zeros(len(nodes)); ind=np.zeros(len(nodes))
        for a,b in arcs: out[a]+=1; ind[b]+=1
        stats.append(dict(trip=tid, components=len(comps),
                          largest_pct=100*max(comps.values())/len(nodes),
                          dead_ends=int((out==0).sum()), sources=int((ind==0).sum())))
pd.DataFrame(stats).describe().round(1)

## 5. GPS noise level

Measured on trip 00000001 as the perpendicular distance from 800 randomly sampled raw GPS points to the nearest ground-truth route arc (equirectangular approximation):
- median **4.2 m**, mean **4.8 m**, 90th percentile **9.7 m**, max **27.2 m**

This is typical consumer-GPS noise. It justifies a Gaussian emission probability in the HMM with sigma in the 5-10 m range, and candidate arcs within roughly a 25-30 m search radius per point.

In [ ]:
def point_seg_dist_m(p, a, b):
    kx = 111320.0*math.cos(math.radians(p[1])); ky = 110540.0
    pa=np.array([(a[0]-p[0])*kx,(a[1]-p[1])*ky]); pb=np.array([(b[0]-p[0])*kx,(b[1]-p[1])*ky])
    ba=pb-pa; L2=ba@ba
    if L2==0: return float(pa@pa)**.5
    t=max(0.0,min(1.0,-(pa@ba)/L2)); proj=pa+t*ba
    return float(proj@proj)**.5

tid="00000001"; d=os.path.join(DATA,"train",tid)
t,_,_=read_track(os.path.join(d,tid+".track"))
nodes=read_nodes(os.path.join(d,tid+".nodes")); arcs=read_arcs(os.path.join(d,tid+".arcs"))
route=read_route(os.path.join(d,tid+".route"))
rng=np.random.default_rng(0)
pairs=[(nodes[arcs[i][0]],nodes[arcs[i][1]]) for i in route]
dists=[min(point_seg_dist_m(t[i],a,b) for a,b in pairs) for i in rng.choice(len(t),800,replace=False)]
dists=np.array(dists)
print(pd.Series(dists).describe().round(2))
plt.figure(figsize=(8,4.5)); plt.hist(dists,bins=40,color="purple")
plt.title("Distance from raw GPS point to ground-truth route (m)"); plt.xlabel("m"); plt.ylabel("points"); plt.show()

## 5b. GPS coordinate distributions

Latitude and longitude histograms over all 227,577 GPS points. The points cluster around the trip cities (strong European cluster around 45-60 N, secondary clusters in North America and Asia). This confirms coordinates are stored as plain WGS84 degrees in the `.track` files (`longitude latitude timestamp` order) and that per-trip metric projections are required.

In [ ]:
lats, lons = [], []
for split in ["train", "student_test"]:
    for d in trip_dirs(split):
        tid = os.path.basename(d)
        t, _, _ = read_track(os.path.join(d, tid + ".track"))
        lats.append(t[:, 1]); lons.append(t[:, 0])
lats = np.concatenate(lats); lons = np.concatenate(lons)
fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
axs[0].hist(lats, bins=60, color="steelblue"); axs[0].set_title(f"Latitude distribution ({len(lats):,} GPS points)")
axs[1].hist(lons, bins=60, color="seagreen");  axs[1].set_title("Longitude distribution")
plt.tight_layout(); plt.show()

## 6. Visualisations

In [ ]:
# Trip locations worldwide + one example raw trajectory
centers=[]
for split in ["train","student_test"]:
    for d in trip_dirs(split):
        tid=os.path.basename(d); t,_,_=read_track(os.path.join(d,tid+".track"))
        centers.append((split,t[:,0].mean(),t[:,1].mean()))
fig,axs=plt.subplots(1,2,figsize=(13,5.5))
tr=[c for c in centers if c[0]=="train"]; te=[c for c in centers if c[0]=="student_test"]
axs[0].scatter([c[1] for c in tr],[c[2] for c in tr],s=25,label="train (80)")
axs[0].scatter([c[1] for c in te],[c[2] for c in te],s=40,marker="^",color="crimson",label="student_test (10)")
axs[0].set_title("Trip locations (mean GPS position per trip)"); axs[0].set_xlabel("Longitude"); axs[0].set_ylabel("Latitude"); axs[0].legend(); axs[0].grid(alpha=.3)
tid="00000001"; t,_,_=read_track(os.path.join(DATA,"train",tid,tid+".track"))
axs[1].plot(t[:,0],t[:,1],"-",lw=.8)
axs[1].plot(t[0,0],t[0,1],"g^",ms=10,label="start"); axs[1].plot(t[-1,0],t[-1,1],"rs",ms=9,label="end")
axs[1].set_title(f"Example raw GPS trajectory: trip {tid}"); axs[1].legend()
axs[1].set_aspect("equal",adjustable="datalim"); plt.show()

In [ ]:
# Road network of the example trip (subsampled for drawing speed)
tid="00000001"; d=os.path.join(DATA,"train",tid)
nodes=read_nodes(os.path.join(d,tid+".nodes")); arcs=read_arcs(os.path.join(d,tid+".arcs"))
fig,ax=plt.subplots(figsize=(8,8))
step=max(1,len(arcs)//60000)
for a,b in arcs[::step]:
    ax.plot([nodes[a,0],nodes[b,0]],[nodes[a,1],nodes[b,1]],"-",lw=.3,color="gray",alpha=.5)
ax.set_title(f"Road network of trip {tid}: {len(nodes):,} nodes, {len(arcs):,} arcs (subsampled)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.set_aspect("equal",adjustable="datalim"); plt.show()

In [ ]:
# Noisy GPS overlaid on the network with the ground-truth route
route=read_route(os.path.join(d,tid+".route"))
t,_,_=read_track(os.path.join(d,tid+".track"))
fig,ax=plt.subplots(figsize=(8,8))
for a,b in arcs[::step]:
    ax.plot([nodes[a,0],nodes[b,0]],[nodes[a,1],nodes[b,1]],"-",lw=.3,color="lightgray",alpha=.6)
ax.plot(t[:,0],t[:,1],".",ms=2.5,color="steelblue",label="noisy GPS points")
rt=[]
for aid in route:
    a,b=arcs[aid]; rt.append(nodes[a]); rt.append(nodes[b])
rt=np.array(rt)
ax.plot(rt[:,0],rt[:,1],"-",lw=1.6,color="crimson",label="ground-truth route")
ax.set_title(f"Trip {tid}: noisy GPS vs road network vs ground truth"); ax.legend()
ax.set_aspect("equal",adjustable="datalim"); plt.show()

In [ ]:
# Dataset distributions
fig,axs=plt.subplots(2,2,figsize=(11,8))
axs[0,0].hist(df.gps_points,bins=20,color="steelblue"); axs[0,0].set_title("GPS points per trip")
axs[0,1].hist(df.nodes,bins=20,color="seagreen");      axs[0,1].set_title("Nodes per trip")
axs[1,0].hist(df.arcs,bins=20,color="darkorange");     axs[1,0].set_title("Arcs per trip")
axs[1,1].hist(df.route_len.dropna(),bins=20,color="crimson"); axs[1,1].set_title("Route length (arcs) per trip")
for a in axs.flat: a.set_ylabel("trips")
plt.tight_layout(); plt.show()

## 7. Interactive Folium map

Road network (subsampled), noisy GPS points and the ground-truth route for trip 00000001, rendered on OpenStreetMap tiles. Saved as `folium_map.html`.

In [ ]:
import folium
center=[float(t[:,1].mean()),float(t[:,0].mean())]
m=folium.Map(location=center,zoom_start=13,tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}", attr="Tiles &copy; Esri - Source: Esri, DeLorme, NAVTEQ, USGS, Intermap, iPC, NRCAN, Esri Japan, METI, Esri China (Hong Kong), Esri (Thailand), TomTom, 2012")
folium.PolyLine([[float(p[1]),float(p[0])] for p in rt],color="red",weight=3,tooltip="ground truth").add_to(m)
fg=folium.FeatureGroup(name="noisy GPS")
for p in t[::2]:
    folium.CircleMarker([float(p[1]),float(p[0])],radius=2,color="blue",fill=True,fill_opacity=.7).add_to(fg)
fg.add_to(m); folium.LayerControl().add_to(m)
m.save("folium_map.html")
m

## 7b. Folium maps for several trips

`folium_map_multi_trip.html` shows four geographically spread trips (raw GPS trace coloured, ground-truth route dashed black) as toggleable layers - zoom into any of them to inspect noise, parallel roads and gaps. `folium_map.html` keeps the detailed single-trip view (network + GPS + ground truth for trip 00000001).

In [ ]:
import folium
picks = ["00000001"]  # add more trip ids here, e.g. spread across regions
colors = ["red", "darkblue", "darkgreen", "purple"]
m = folium.Map(location=[45, 20], zoom_start=4, tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}", attr="Tiles &copy; Esri - Source: Esri, DeLorme, NAVTEQ, USGS, Intermap, iPC, NRCAN, Esri Japan, METI, Esri China (Hong Kong), Esri (Thailand), TomTom, 2012")
for tid, color in zip(picks, colors):
    for split in ["train", "student_test"]:
        d = os.path.join(DATA, split, tid)
        if os.path.isdir(d): break
    t, _, _ = read_track(os.path.join(d, tid + ".track"))
    nodes = read_nodes(os.path.join(d, tid + ".nodes")); arcs = read_arcs(os.path.join(d, tid + ".arcs"))
    route = read_route(os.path.join(d, tid + ".route"))
    fg = folium.FeatureGroup(name=f"trip {tid}")
    folium.PolyLine([[float(p[1]), float(p[0])] for p in t], color=color, weight=2, opacity=.8).add_to(fg)
    rt = []
    for aid in route:
        a, b = arcs[aid]; rt.append(nodes[a]); rt.append(nodes[b])
    rt = np.array(rt)
    folium.PolyLine([[float(p[1]), float(p[0])] for p in rt], color="black", weight=2, opacity=.6, dash_array="6").add_to(fg)
    fg.add_to(m)
folium.LayerControl().add_to(m)
m.save("folium_map_multi_trip.html")
m

## 8. Key findings

1. **90 self-contained trips** (80 train / 10 test); each trip has its own road network sized to the trace region (299-155k nodes, 649-307k directed arcs).
2. **227,577 GPS points total**; trips range 262-19,313 points (median 1,496).
3. **Data is clean**: 0 malformed lines, 0 invalid coordinates, 0 non-monotonic timestamps, only 6 duplicate fixes.
4. **1 Hz claim verified**: 96.7% of sampling intervals are exactly 1 s; rare large gaps exist (max ~42 h) and must be handled.
5. **Networks are mostly connected**: largest weak component covers ~98% of nodes (median); few dead ends.
6. **GPS noise is modest**: median 4.2 m / p90 9.7 m / max 27 m from the true route - a Gaussian emission model with sigma ~5-10 m and a ~30 m candidate radius is appropriate.
7. **Ground-truth routes are valid** arc sequences in every trip; test trips in this copy also carry `.route` files (flag to organizers).
8. Trips span **many cities worldwide**, so no single coordinate system assumption holds; all distance work must be done in meters (equirectangular/haversine), not degrees.